# 01 | Framing, data and my baseline

**My user:** an IT support engineer with incomplete incident information. **My question:** can Scout find useful published incidents and help the engineer choose the next check without presenting a guess as a diagnosis?

The user gives Scout symptoms, then adds observations as the investigation develops. Scout returns possible historical matches, checked next steps and a saved case.

**LLM track.** I chose **Hit@5** for retrieval: is the expected incident among the first five results? A wrong or missing source can send someone in the wrong direction. I assess Llama's answer correctness, citations and refusals separately in Notebook 03.

I did not train a new language model.

## 1. Read the real data

I downloaded `incidents.csv` from [SevAnalysis](https://sevanalysis.com/data/) (CC BY-NC 4.0). One row is a published vendor incident, **not** a private IT ticket or a verified fix. I keep the original CSV outside Git.

This cell reads my local copy when available. The committed EDA figures below are from the actual dataset.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import json

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.incident_assistant import LOCAL_CSV, load_incidents, prepare_incidents

if LOCAL_CSV.is_file():
    raw = load_incidents(LOCAL_CSV)
    records = prepare_incidents(raw)
    print(f"Raw rows: {len(raw):,}; searchable: {len(records):,}")
    display(records[["vendor", "title", "started_at", "source_url"]].head(5))
else:
    records = None
    print("Download incidents.csv from SevAnalysis to rerun the cleaning.")

## 2. Clean the records and check their quality

I kept the incident title, vendor, date, impact, available components and original link. I removed unusable core records and exact duplicates. The executed [EDA notebook](06_source_eda.ipynb) contains the full checks; here is the evidence that affected my retrieval design.

**My document choices:** one short status incident is one searchable document. I don't split a six-word title into chunks. I keep title, vendor and components for matching; source ID and a stable INC ID for traceability; start/resolution times, impact and status for context; and the source URL for checking. The derived `search_text` repeats the title to give it more weight. I don't use model-enriched cause labels as ground truth. I mask common email and phone patterns in text, although that is not a full personal-data check.

In [ ]:
# Use the actual results saved after running Notebook 06 on the downloaded CSV.
metrics = json.loads((ROOT / "reports/source_eda_metrics.json").read_text(encoding="utf-8"))
display(pd.DataFrame([
    ["Original rows", metrics["raw_rows"]],
    ["After cleaning", metrics["prepared_rows"]],
    ["Duplicates removed", metrics["duplicate_usable_rows"]],
    ["Vendors", metrics["unique_vendors"]],
    ["Missing components (%)", metrics["missing_pct_prepared"]["Affected components"]],
    ["Median words in a title", metrics["title_length_words"]["median"]],
], columns=["What I checked", "Measured result"]))

### What I learnt from the figures

![Missing fields in source records](../reports/source_eda_missingness.png)

**72.94%** of incidents omit the affected component. I cannot rely on component categories, so Scout searches the text and retains each original link.

![Length of incident titles](../reports/source_eda_title_lengths.png)

The median title is **six words**. This explains why synonyms and abbreviations can cause search misses.

![Published reports by vendor](../reports/source_eda_vendor_coverage.png)

The top 12 vendors account for **36.23%** of records. This tells me about dataset coverage, not which vendor is less reliable.

**Decision:** compare a simple keyword search with TF-IDF (which gives more weight to distinctive words and word pairs). Both search the same prepared records.

## 3. Prepare my evaluation questions

I have **12 questions** with expected published incidents and **three unanswerable questions**, such as asking for internal information the public source could not contain.

**Important:** I created and reworded these after work on Scout had started. I checked IDs against the downloaded CSV but have not independently approved every vendor page. This is a **retrospective, provisional test**, not a truly untouched pre-build evaluation set.

In [ ]:
questions = pd.read_csv(
    ROOT / "evals/incident_eval_questions_provisional.csv",
    keep_default_na=False,
)
questions["answerable"] = questions["answerable"].astype(str).str.lower().eq("true")
print("Total:", len(questions), "Answerable:", int(questions["answerable"].sum()),
      "Should refuse:", int((~questions["answerable"]).sum()))
display(questions[["question_id", "question", "answerable"]])

## 4. Compare with a simple baseline

I tested **keyword overlap** against **TF-IDF** on the same 12 answerable questions. Hit@5 means the expected historical incident appears in the top five. Timing is measured separately and excludes index construction.

I used the saved results so they are visible in GitHub even without the downloaded CSV.

In [ ]:
baseline = pd.read_csv(ROOT / "reports/retrieval_baseline_2026-11.csv")
for column in ["keyword_hit_at_5", "tfidf_hit_at_5"]:
    baseline[column] = baseline[column].astype(str).str.lower().eq("true")

for name, column in [
    ("Keyword overlap", "keyword_hit_at_5"),
    ("TF-IDF", "tfidf_hit_at_5"),
]:
    hits = int(baseline[column].sum())
    print(f"{name}: {hits}/{len(baseline)} expected incidents in top five")

![Recorded retrieval Hit@5](../reports/retrieval_hit_at_5.svg)

**What it tells me:** keyword overlap found **9/12**, while TF-IDF found **8/12**. TF-IDF was faster (0.007 vs 0.551 seconds per query after indexing), but it did **not** beat the baseline on relevance in this small test. I kept it as a fast prototype method, documented the misses and evaluated the LLM separately in [Notebook 03](03_evaluation.ipynb).